# Solar Filament Segmentation Challenge 2026 - Full-Resolution Preprocessing

This notebook trains and evaluates a YOLO26 segmentation model on the original 2048 x 2048 images while staying within a 30 GiB RAM and 57.6 GiB disk budget.

## Resource settings

- **2048 x 2048 input resolution** for training, validation, and test inference.
- **Both available T4 GPUs** for training, with a global batch of 2 (one image per GPU).
- **At most four CPU data workers** during training (two per GPU); CPU preprocessing pools also use at most four single-threaded workers.
- **Adaptive disk cache**: use it only when the actual working volume can fit it with headroom; otherwise disable caching. RAM caching is never used.
- **One image per inference batch** and a bounded validation queue to keep full-resolution masks from accumulating.
- Submission CSV validation reads rows incrementally rather than loading all RLE strings into RAM.

Validation metrics include PQ, SQ, RQ, Dice, IoU, precision, recall, and instance error counts. Test ground truth is never accessed.

## Added experiment options

Set `PREPROCESSING_METHOD` to `"raw"`, `"clahe"`, `"percentile"`, or `"gamma"`, then
run all cells for that method. These are the intensity transforms from the
native-2048 notebook. `raw` is the control with merged annotators. Each method has
its own dataset, checkpoints, metrics and submission in `preprocessing_<method>_union/`.
The selected transform is applied once to training, validation and test images;
all three model channels receive the same processed grayscale intensities.

For each training/YOLO-validation filename, all annotator masks are combined by
pixelwise union before the existing 8-connected-component/polygon conversion.
There is one YOLO sample per physical filename. The original filename split,
training call (including the batch size, rotations and minimum component area),
final per-annotator evaluation, and submission logic are retained.

Five distinct images are previewed at each stage: annotator merge/training targets,
all intensity transforms, validation postprocessing, test inputs, and masks decoded
from the written submission CSV. Figures are displayed inline and saved under
`previews/`. Set `PREVIEW_COUNT` to change the count. Plotting does not change targets,
predictions, confidence selection, or training settings.


## 1. Install dependencies and import libraries

The training configuration uses disk caching rather than a RAM cache. Package installation disables pip's wheel cache to conserve the session disk.


In [ ]:
%pip -q install ultralytics pycocotools scikit-learn tqdm torchmetrics threadpoolctl psutil

In [ ]:
from pathlib import Path
import os
import csv
import gc
import random
import shutil
import time
import warnings
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    import psutil
except ImportError:
    psutil = None

from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from torchmetrics.segmentation import DiceScore
from threadpoolctl import threadpool_limits
from ultralytics import YOLO

# Hide standard Python warnings
warnings.filterwarnings("ignore")

# Hide underlying C-level / ctypes environment logs
os.environ["PYTHONWARNINGS"] = "ignore"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_DEVICE_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
MAX_GPUS_TO_USE = 2
GPU_DEVICE_IDS = list(range(min(CUDA_DEVICE_COUNT, MAX_GPUS_TO_USE)))
GPU_COUNT_USED = len(GPU_DEVICE_IDS)

if GPU_COUNT_USED >= 2:
    TRAIN_DEVICE = GPU_DEVICE_IDS
elif GPU_COUNT_USED == 1:
    TRAIN_DEVICE = GPU_DEVICE_IDS[0]
else:
    TRAIN_DEVICE = "cpu"

# Ultralytics prediction uses one explicit device; distributed training below
# uses both GPUs when they are available.
INFERENCE_DEVICE = GPU_DEVICE_IDS[0] if GPU_COUNT_USED else "cpu"

print(f"Training device(s): {TRAIN_DEVICE}")
print(f"Inference device: {INFERENCE_DEVICE}")
if CUDA_DEVICE_COUNT:
    for gpu_id in GPU_DEVICE_IDS:
        print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
    print(f"CUDA devices available: {CUDA_DEVICE_COUNT}; using {GPU_COUNT_USED}")

CPU_COUNT = os.cpu_count() or 1
print(f"Logical CPU count: {CPU_COUNT}")

## 2. Configure paths and experiment

The notebook caps CPU worker processes at four. Multi-GPU training uses two data workers per GPU (four total); custom preprocessing stages use a four-process pool with one thread per worker. Images and inference masks remain at 2048 x 2048, while batches and queues stay small to protect the 30 GiB RAM limit.


In [ ]:
SEED = 42

NUM_EPOCHS = 150

VALIDATION_FRACTION = 0.2
PATIENCE = 20

IMAGE_SIZE = 2048
NUM_WORKERS = 4  # Hard cap for custom CPU worker pools.
WORKER_CPU_THREADS = 1

# In Ultralytics DDP, workers is per GPU rank. Keep the global worker count <= 4.
YOLO_WORKERS_PER_GPU = (
    min(NUM_WORKERS // GPU_COUNT_USED, 4) if GPU_COUNT_USED > 0 else 0
)
BATCH_SIZE = GPU_COUNT_USED if GPU_COUNT_USED > 0 else 1  # One image per GPU.

# One native-resolution image at a time keeps full-size masks bounded.
VAL_BATCH_SIZE = 1
MAX_PENDING_VALIDATION = NUM_WORKERS * 2
MAX_DETECTIONS = 100

RAM_LIMIT_GIB = 30.0
RAM_RESERVE_GIB = 6.0
DISK_LIMIT_GIB = 57.6
DISK_RESERVE_GIB = (
    10.0  # Preferred reserve; reduced only if the mounted volume requires it.
)
MIN_DISK_RESERVE_GIB = 4.0
CACHE_SAFETY_FACTOR = 1.25
TRAIN_OUTPUT_RESERVE_GIB = 2.0
GIB = 1024**3
YOLO_CACHE_MODE = "disk"  # Preflight switches this to False if disk cache cannot fit.
ACTIVE_DISK_RESERVE_GIB = DISK_RESERVE_GIB

MIN_COMPONENT_AREA = 20
YOLO_MODEL_NAME = "yolo26s-seg.pt"

# Same intensity-only methods/settings as the native-2048 notebook.
# Run all cells separately for each method; each starts from YOLO_MODEL_NAME.
PREPROCESSING_METHOD = "raw"  # "raw", "clahe", "percentile", or "gamma"
PREPROCESSING_CONFIG = {
    "clahe_clip": 2.0,
    "clahe_grid": (8, 8),
    "percentiles": (1.0, 99.0),
    "gamma": 0.8,
}
if PREPROCESSING_METHOD not in {"raw", "clahe", "percentile", "gamma"}:
    raise ValueError(f"Unknown preprocessing method: {PREPROCESSING_METHOD}")

EXPERIMENT_DIR = Path.cwd() / f"preprocessing_{PREPROCESSING_METHOD}_union"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
YOLO_DATA_DIR = EXPERIMENT_DIR / "yolo_filament_dataset"
RUNS_DIR = EXPERIMENT_DIR / "yolo_runs"
RUN_NAME = "yolo26s_filament_full_resolution"
DATA_YAML = YOLO_DATA_DIR / "filament.yaml"

SUBMISSION_PATH = EXPERIMENT_DIR / "solar_filament_yolo26s_submission.csv"
FINAL_METRICS_PATH = EXPERIMENT_DIR / "solar_filament_yolo26s_validation_metrics.csv"
PER_IMAGE_METRICS_PATH = EXPERIMENT_DIR / "solar_filament_yolo26s_per_image_metrics.csv"

PREVIEW_COUNT = 5  # Distinct physical images per gallery.
PREVIEW_DIR = EXPERIMENT_DIR / "previews"

REBUILD_YOLO_DATASET = True
CREATE_SUBMISSION = True

CONFIDENCE_CANDIDATES = (0.15, 0.25, 0.35, 0.45, 0.55)
PQ_MATCH_IOU = 0.50


def _read_cgroup_memory_pair(limit_path, current_path):
    try:
        limit_text = Path(limit_path).read_text().strip()
        current_text = Path(current_path).read_text().strip()
        if limit_text == "max":
            return None
        limit_bytes = int(limit_text)
        current_bytes = int(current_text)
        if limit_bytes >= (1 << 60):  # cgroup v1 unlimited sentinel
            return None
        return max(0, limit_bytes - current_bytes)
    except (OSError, ValueError):
        return None


def available_ram_bytes():
    for limit_path, current_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        (
            "/sys/fs/cgroup/memory/memory.limit_in_bytes",
            "/sys/fs/cgroup/memory/memory.usage_in_bytes",
        ),
    ):
        available = _read_cgroup_memory_pair(limit_path, current_path)
        if available is not None:
            return available
    if psutil is not None:
        try:
            return int(psutil.virtual_memory().available)
        except Exception:
            pass
    return None


def require_ram_headroom(stage, minimum_gib):
    available = available_ram_bytes()
    if available is None:
        print(
            f"[RAM] Could not read a runtime memory limit before {stage}; using bounded batches/workers."
        )
        return
    available_gib = available / GIB
    print(f"[RAM] Available before {stage}: {available_gib:.1f} GiB")
    if available_gib < minimum_gib:
        raise MemoryError(
            f"Only {available_gib:.1f} GiB RAM is available before {stage}; "
            f"need at least {minimum_gib:.1f} GiB to preserve headroom under the "
            f"{RAM_LIMIT_GIB:.1f} GiB session limit. Free memory or restart the session."
        )


def directory_size_bytes(root):
    root = Path(root)
    if not root.exists():
        return 0
    total = 0
    for item in root.rglob("*"):
        if item.is_file() and not item.is_symlink():
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return total


def preflight_training_resources(train_split, val_split, image_dir):
    global YOLO_CACHE_MODE, ACTIVE_DISK_RESERVE_GIB

    require_ram_headroom("dataset preparation", minimum_gib=RAM_RESERVE_GIB + 2.0)
    records = pd.concat([train_split, val_split], ignore_index=True).drop_duplicates("file_name")

    # Conservative cache estimate: RGB uint8 array per physical image at full size.
    estimated_cache_bytes = int(
        len(records) * IMAGE_SIZE * IMAGE_SIZE * 3 * CACHE_SAFETY_FACTOR
    )
    # Processed images are lossless grayscale PNGs, one per physical filename.
    # Include encoding overhead even for nearly incompressible 2048x2048 images.
    copy_bytes = int(len(records) * IMAGE_SIZE * IMAGE_SIZE * 1.10)

    output_reserve = int(TRAIN_OUTPUT_RESERVE_GIB * GIB)
    workspace_bytes = directory_size_bytes(Path.cwd())
    disk = shutil.disk_usage(YOLO_DATA_DIR.parent)
    effective_budget_bytes = min(int(DISK_LIMIT_GIB * GIB), int(disk.total))

    # Prefer disk caching with the full reserve. If the mounted filesystem is
    # smaller than the advertised session quota, turn caching off (never RAM-on)
    # and only reduce the reserve as far as 4 GiB.
    cache_options = (("disk", estimated_cache_bytes), (False, 0))
    reserve_options = [
        value
        for value in (DISK_RESERVE_GIB, 8.0, 6.0, MIN_DISK_RESERVE_GIB)
        if value <= DISK_RESERVE_GIB
    ]
    selected_plan = None

    for cache_mode, cache_bytes in cache_options:
        for reserve_gib in reserve_options:
            reserve_bytes = int(reserve_gib * GIB)
            planned_new_bytes = (
                cache_bytes + copy_bytes + output_reserve + reserve_bytes
            )
            if (
                planned_new_bytes <= effective_budget_bytes
                and workspace_bytes + planned_new_bytes <= effective_budget_bytes
                and disk.free >= planned_new_bytes
            ):
                selected_plan = (
                    cache_mode,
                    reserve_gib,
                    planned_new_bytes,
                    cache_bytes,
                )
                break
        if selected_plan is not None:
            break

    if selected_plan is None:
        no_cache_minimum = copy_bytes + output_reserve + int(MIN_DISK_RESERVE_GIB * GIB)
        raise OSError(
            "Insufficient writable disk even with YOLO image caching disabled. "
            f"Filesystem capacity: {disk.total / GIB:.1f} GiB; free: {disk.free / GIB:.1f} GiB; "
            f"workspace: {workspace_bytes / GIB:.1f} GiB; minimum planned use: "
            f"{no_cache_minimum / GIB:.1f} GiB (includes a {MIN_DISK_RESERVE_GIB:.1f} GiB reserve). "
            "Free disk space or select a larger working volume."
        )

    YOLO_CACHE_MODE, ACTIVE_DISK_RESERVE_GIB, planned_new_bytes, cache_bytes = (
        selected_plan
    )
    print(
        f"[DISK] Cache mode: {YOLO_CACHE_MODE}; estimated cache: {cache_bytes / GIB:.1f} GiB; "
        f"worst-case image copies: {copy_bytes / GIB:.2f} GiB."
    )
    print(
        f"[DISK] Planned additional use: {planned_new_bytes / GIB:.1f} GiB; "
        f"reserved free space: {ACTIVE_DISK_RESERVE_GIB:.1f} GiB; "
        f"filesystem free: {disk.free / GIB:.1f} GiB."
    )
    if YOLO_CACHE_MODE is False:
        print(
            "[DISK] Full-resolution training will continue without image caching; RAM caching is disabled."
        )


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_main_process_threads():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass


configure_main_process_threads()
set_seed()

# --- Logging Framework Configurations ---
print(f"Random seed initialization: {SEED}")
print(
    f"Split ratio: Train = {1.0 - VALIDATION_FRACTION:.2f} / Validation = {VALIDATION_FRACTION:.2f}"
)
print(f"Training epochs: {NUM_EPOCHS} (Early stopping patience: {PATIENCE})")
print(f"Input size: {IMAGE_SIZE} x {IMAGE_SIZE}")
print(f"Custom CPU worker cap: {NUM_WORKERS}")
print(f"YOLO data workers per GPU: {YOLO_WORKERS_PER_GPU}")
print(f"YOLO global batch size: {BATCH_SIZE}")
print(f"Inference batch size: {VAL_BATCH_SIZE}")
print("Image cache mode: selected by disk preflight; RAM caching is disabled.")
print(f"Preprocessing method: {PREPROCESSING_METHOD}; settings: {PREPROCESSING_CONFIG}")
print(f"Experiment directory: {EXPERIMENT_DIR}")


In [ ]:
def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]

    for parent in candidates:
        if not parent.exists():
            continue

        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train" / "train_images").is_dir() and (
                root / "test" / "test_images"
            ).is_dir():
                return root

    raise FileNotFoundError(
        "Could not find MAGFiLO_1.0_Kaggle_2026 with train and test image folders."
    )


DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"YOLO model: {YOLO_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Global training batch: {BATCH_SIZE} ({YOLO_WORKERS_PER_GPU} workers per GPU)")
print(f"Validation fraction: {VALIDATION_FRACTION:.0%}")
print(f"PQ matching IoU threshold: {PQ_MATCH_IOU:.2f}")

## 3. Load annotations and create a leakage-safe split

The split is performed using the physical filename rather than the COCO record ID. Multiple annotation records for the same JPEG remain on one side of the split.


In [ ]:
coco = COCO(str(ANNOTATION_PATH))
image_ids = list(coco.imgs.keys())
records = pd.DataFrame(
    [
        {"image_id": image_id, "file_name": coco.imgs[image_id]["file_name"]}
        for image_id in image_ids
    ]
)

splitter = GroupShuffleSplit(
    n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED
)
train_idx, val_idx = next(splitter.split(records, groups=records["file_name"]))

train_records = records.iloc[train_idx].reset_index(drop=True)
val_records = records.iloc[val_idx].reset_index(drop=True)

train_files = set(train_records["file_name"])
val_files = set(val_records["file_name"])

assert train_files.isdisjoint(val_files), "A physical image appears in both splits."

print(
    f"Training records: {len(train_records):,} across {len(train_files):,} physical images"
)
print(
    f"Validation records: {len(val_records):,} across {len(val_files):,} physical images"
)
print(f"Train/validation filename overlap: {len(train_files & val_files)}")

## 4. Build the YOLO segmentation dataset

All COCO image records with the same physical filename contribute to one pixelwise
mask union, following the native-2048 training-target merge. The existing
8-connected-component, minimum-area and YOLO-polygon logic then builds labels once
per filename. Processed grayscale images are saved as lossless PNGs, with the same
intensity transform used at validation/test inference. Disk preflight counts one
PNG and one optional RGB cache per physical image.


In [ ]:
def preprocess_grayscale(image, method=PREPROCESSING_METHOD, config=None):
    """Native-2048 intensity transforms, preserving the grayscale pixel grid."""
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Preprocessing expects uint8 grayscale")
    config = PREPROCESSING_CONFIG if config is None else config
    if method == "raw":
        result = image.copy()
    elif method == "clahe":
        result = cv2.createCLAHE(
            clipLimit=config["clahe_clip"], tileGridSize=tuple(config["clahe_grid"])
        ).apply(image)
    elif method == "percentile":
        # Ignore black background when estimating intensity bounds.
        pixels = image[image > 0]
        low, high = np.percentile(pixels, config["percentiles"]) if pixels.size else (0, 0)
        result = (
            np.clip((image.astype(np.float32) - low) * 255 / (high - low), 0, 255)
            .astype(np.uint8)
        ) if high > low else image.copy()
    elif method == "gamma":
        table = np.round(255 * (np.arange(256) / 255) ** config["gamma"]).astype(np.uint8)
        result = cv2.LUT(image, table)
    else:
        raise ValueError(f"Unknown preprocessing method: {method}")
    return np.ascontiguousarray(result)


def merged_annotation_mask(coco_api, image_ids):
    """Union every annotation from every annotator record for one filename."""
    first = coco_api.imgs[image_ids[0]]
    shape = (first["height"], first["width"])
    merged = np.zeros(shape, dtype=np.uint8)
    for image_id in image_ids:
        metadata = coco_api.imgs[image_id]
        if metadata["file_name"] != first["file_name"] or (
            metadata["height"], metadata["width"]
        ) != shape:
            raise ValueError("Annotator records must describe the same image and dimensions")
        annotations = coco_api.loadAnns(coco_api.getAnnIds(imgIds=[image_id]))
        for annotation in annotations:
            merged |= coco_api.annToMask(annotation).astype(np.uint8)
    return merged


def configure_worker_process():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        torch.set_num_interop_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        threadpool_limits(limits=WORKER_CPU_THREADS)
    except Exception:
        pass

    if hasattr(os, "sched_getaffinity"):
        available_cpus = sorted(os.sched_getaffinity(0))
        if available_cpus:
            identity = multiprocessing.current_process()._identity
            worker_index = identity[0] - 1 if identity else 0
            cpu = available_cpus[worker_index % len(available_cpus)]
            if hasattr(os, "sched_setaffinity"):
                os.sched_setaffinity(0, {cpu})


def mask_to_yolo_segments(mask, min_area):
    mask = (mask > 0).astype(np.uint8)
    height, width = mask.shape
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    segments = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue

        component = (labels == label).astype(np.uint8)
        contours, _ = cv2.findContours(
            component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        if len(contour) < 3 or cv2.contourArea(contour) <= 0:
            continue

        # Ensure the component actually has a valid structural bounding box ---
        bx, by, bw, bh = cv2.boundingRect(contour)
        if bw < 2 or bh < 2:  # Rejects single-pixel lines or noise
            continue

        points = contour.reshape(-1, 2).astype(np.float32)
        points[:, 0] /= width
        points[:, 1] /= height
        points = np.clip(points, 0.0, 1.0)

        # Re-verify normalized bounding box dimensions to protect Pillow ---
        if len(points) >= 3:
            x_min, y_min = points[:, 0].min(), points[:, 1].min()
            x_max, y_max = points[:, 0].max(), points[:, 1].max()

            # If the normalization completely flattened a microscopic edge, ignore it
            if (x_max - x_min) <= 1e-4 or (y_max - y_min) <= 1e-4:
                continue

            segments.append(points)

    return segments


def worker_process_record(
    row_dict,
    annotation_path_str,
    min_area,
    train_image_dir_str,
    image_dir_str,
    label_dir_str,
    preprocessing_method,
    preprocessing_config,
):
    global _local_coco
    configure_worker_process()

    if "_local_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_coco = COCO(annotation_path_str)
        sys.stdout = old_stdout

    train_img_dir = Path(train_image_dir_str)
    img_out_dir = Path(image_dir_str)
    lbl_out_dir = Path(label_dir_str)

    image_ids = row_dict["image_ids"]
    image_id = image_ids[0]
    file_name = row_dict["file_name"]
    source_path = train_img_dir / file_name

    if not source_path.is_file():
        return 0

    source_stem = Path(file_name).stem
    sample_stem = f"{image_id}_{source_stem}"

    image_path = img_out_dir / f"{sample_stem}.png"
    label_path = lbl_out_dir / f"{sample_stem}.txt"

    image_path.parent.mkdir(parents=True, exist_ok=True)
    if image_path.exists() or image_path.is_symlink():
        image_path.unlink()

    image = cv2.imread(str(source_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read grayscale image: {source_path}")
    image = preprocess_grayscale(image, preprocessing_method, preprocessing_config)
    if not cv2.imwrite(str(image_path), image):
        raise OSError(f"Could not write processed image: {image_path}")

    mask = merged_annotation_mask(_local_coco, image_ids)
    lines = []
    for points in mask_to_yolo_segments(mask, min_area):
        flat = points.reshape(-1)
        values = ["0"]
        values.extend(f"{value:.6f}" for value in flat)
        lines.append(" ".join(values))

    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return len(lines)


def write_yolo_split_parallel(split_records, split_name):
    print(f"\n[START] Building YOLO split: '{split_name}'", flush=True)
    image_dir = YOLO_DATA_DIR / "images" / split_name
    label_dir = YOLO_DATA_DIR / "labels" / split_name

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    # Preserve the original split, but write exactly one target/image per filename.
    records_list = (
        split_records.groupby("file_name", sort=False)["image_id"]
        .agg(list)
        .reset_index(name="image_ids")
        .to_dict(orient="records")
    )
    worker_task = partial(
        worker_process_record,
        annotation_path_str=str(ANNOTATION_PATH),
        min_area=MIN_COMPONENT_AREA,
        train_image_dir_str=str(TRAIN_IMAGE_DIR),
        image_dir_str=str(image_dir),
        label_dir_str=str(label_dir),
        preprocessing_method=PREPROCESSING_METHOD,
        preprocessing_config=PREPROCESSING_CONFIG,
    )

    empty_labels = 0
    total_instances = 0

    with multiprocessing.Pool(
        processes=NUM_WORKERS, initializer=configure_worker_process
    ) as pool:
        results = tqdm(
            pool.imap_unordered(worker_task, records_list, chunksize=4),
            total=len(records_list),
            desc=f"Preparing YOLO {split_name}",
        )

        for idx, instance_count in enumerate(results, start=1):
            total_instances += instance_count
            if instance_count == 0:
                empty_labels += 1

            if idx == 1 or idx % 100 == 0 or idx == len(records_list):
                print(
                    f"[{split_name.upper()} Pre-Processing] {idx:,}/{len(records_list):,} | polygons: {total_instances:,} | empty: {empty_labels:,}",
                    flush=True,
                )

    print(
        f"[FINISHED] {split_name}: {len(records_list):,} records | {total_instances:,} polygons | empty labels: {empty_labels:,}\n",
        flush=True,
    )


if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    print(f"[CLEANUP] Removing old directory: {YOLO_DATA_DIR}", flush=True)
    shutil.rmtree(YOLO_DATA_DIR)

preflight_training_resources(train_records, val_records, TRAIN_IMAGE_DIR)
write_yolo_split_parallel(train_records, "train")
write_yolo_split_parallel(val_records, "val")

DATA_YAML.write_text(
    f"path: '{YOLO_DATA_DIR.as_posix()}'\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: filament\n",
    encoding="utf-8",
)

print(f"YOLO dataset configuration: {DATA_YAML}")
print(DATA_YAML.read_text())

### Preview helpers

The following plotting helpers save and display figures without changing model inputs, masks, metrics or training settings.


In [ ]:
# Plotting helpers only: no random sampling and no extra model inference.
def select_preview_items(items, count=None):
    items = list(items)
    count = PREVIEW_COUNT if count is None else count
    if not items or count <= 0:
        return []
    indices = np.linspace(0, len(items) - 1, min(count, len(items)), dtype=int)
    return [items[int(index)] for index in indices]


def show_grayscale(axis, image):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")


def show_mask_overlay(axis, image, mask, alpha=0.45):
    show_grayscale(axis, image)
    axis.imshow(
        np.ma.masked_where(~mask.astype(bool), np.ones(mask.shape, dtype=np.uint8)),
        cmap="autumn", vmin=0, vmax=1, alpha=alpha, interpolation="nearest",
    )


def show_component_overlay(axis, image, components):
    show_grayscale(axis, image)
    labels = np.zeros(image.shape, dtype=np.int32)
    for index, component in enumerate(components, start=1):
        labels[component.astype(bool)] = index
    if components:
        axis.imshow(
            np.ma.masked_where(labels == 0, labels),
            cmap=plt.get_cmap("tab20", max(1, len(components))),
            vmin=0.5, vmax=len(components) + 0.5,
            alpha=0.5, interpolation="nearest",
        )
    for index, component in enumerate(components, start=1):
        moments = cv2.moments(component.astype(np.uint8))
        if moments["m00"]:
            axis.text(
                moments["m10"] / moments["m00"], moments["m01"] / moments["m00"],
                f"F{index}\n{int(component.sum()):,} px", ha="center", va="center", fontsize=7,
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.8),
            )


def finish_preview(fig, stage, filename):
    for axis in fig.axes:
        axis.axis("off")
    destination = PREVIEW_DIR / stage / f"{Path(filename).stem}.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(rect=(0, 0, 1, 0.95), h_pad=3.0, w_pad=1.5)
    try:
        fig.savefig(destination, dpi=110, bbox_inches="tight")
        plt.show()
    finally:
        plt.close(fig)
    print(f"Saved preview: {destination}")


def read_preview_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"Could not read preview image: {path}")
    return image


def preview_training_sources():
    # Prefer multi-annotator sources, keeping each physical image only once.
    grouped = train_records.groupby("file_name", sort=False)["image_id"].agg(list).reset_index()
    grouped["annotators"] = grouped["image_id"].map(len)
    grouped = grouped.sort_values(["annotators", "file_name"], ascending=[False, True])
    return grouped.head(PREVIEW_COUNT).to_dict(orient="records")


def preview_polygon_mask(label_path, shape):
    mask = np.zeros(shape, dtype=np.uint8)
    height, width = shape
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        points = np.asarray(line.split()[1:], dtype=np.float32).reshape(-1, 2)
        pixels = np.round(points * np.array([width, height])).astype(np.int32)
        cv2.fillPoly(mask, [pixels], 1)
    return mask.astype(bool)


def preview_annotator_merges():
    for row in preview_training_sources():
        name, ids = row["file_name"], row["image_id"]
        image = read_preview_image(TRAIN_IMAGE_DIR / name)
        merged = merged_annotation_mask(coco, ids).astype(bool)
        fig, axes = plt.subplots(2, 3, figsize=(16, 11))
        axes = axes.ravel()
        show_grayscale(axes[0], image)
        axes[0].set_title("Original grayscale image")
        show_grayscale(axes[1], image)
        colors = plt.get_cmap("tab10")
        for index, image_id in enumerate(ids):
            mask = merged_annotation_mask(coco, [image_id]).astype(bool)
            color = colors(index % 10)
            if mask.any():
                axes[1].contour(mask.astype(np.uint8), levels=[0.5], colors=[color], linewidths=0.7)
            axes[1].plot([], [], color=color, label=f"Annotator record {index + 1}")
        axes[1].legend(loc="lower left", fontsize=7)
        axes[1].set_title(f"Before merge: {len(ids)} annotator records")
        axes[2].imshow(merged, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[2].set_title("After merge: pixelwise union mask")
        show_mask_overlay(axes[3], image, merged)
        axes[3].set_title("Merged mask over original image")
        sample_stem = f"{ids[0]}_{Path(name).stem}"
        processed = read_preview_image(YOLO_DATA_DIR / "images" / "train" / f"{sample_stem}.png")
        show_grayscale(axes[4], processed)
        axes[4].set_title(f"Saved training image: {PREPROCESSING_METHOD}")
        polygon_mask = preview_polygon_mask(
            YOLO_DATA_DIR / "labels" / "train" / f"{sample_stem}.txt", image.shape,
        )
        show_mask_overlay(axes[5], processed, polygon_mask)
        axes[5].set_title(f"YOLO polygon target overlay (min area {MIN_COMPONENT_AREA})")
        fig.suptitle(f"Annotation merge / training target | {name}", fontsize=13)
        finish_preview(fig, "merged_masks", name)


def preview_preprocessing():
    for row in preview_training_sources():
        name = row["file_name"]
        image = read_preview_image(TRAIN_IMAGE_DIR / name)
        merged = merged_annotation_mask(coco, row["image_id"]).astype(bool)
        fig, axes = plt.subplots(2, 3, figsize=(16, 11))
        axes = axes.ravel()
        for axis, method in zip(axes[:4], ("raw", "clahe", "percentile", "gamma")):
            processed = preprocess_grayscale(image, method, PREPROCESSING_CONFIG)
            show_grayscale(axis, processed)
            axis.set_title(f"{method}" + (" (selected)" if method == PREPROCESSING_METHOD else ""))
        show_mask_overlay(axes[4], image, merged)
        axes[4].set_title("Before preprocessing: raw + merged mask")
        show_mask_overlay(axes[5], preprocess_grayscale(image), merged)
        axes[5].set_title(f"After preprocessing: {PREPROCESSING_METHOD} + same mask")
        fig.suptitle(f"Intensity preprocessing | {name} | fixed display range 0-255", fontsize=13)
        finish_preview(fig, "preprocessing", name)


def preview_validation_postprocessing(examples):
    for example in examples[:PREVIEW_COUNT]:
        image, instances = example["image"], example["instances"]
        before = union_instances(instances, 0.0)
        selected = union_instances(instances, BEST_CONFIDENCE)
        before = np.zeros(image.shape, bool) if before is None else before
        selected = np.zeros(image.shape, bool) if selected is None else selected
        components = build_final_prediction_instances(instances, BEST_CONFIDENCE)
        final_mask = np.zeros(image.shape, dtype=bool)
        for component in components:
            final_mask |= component
        fig, axes = plt.subplots(2, 3, figsize=(16, 11))
        axes = axes.ravel()
        show_grayscale(axes[0], image)
        axes[0].set_title("Original validation image")
        show_mask_overlay(axes[1], image, example["target"])
        axes[1].set_title("Evaluation ground-truth overlay (one annotator)")
        show_mask_overlay(axes[2], image, before)
        axes[2].set_title(f"Before: all {len(instances)} returned model masks")
        show_mask_overlay(axes[3], image, selected)
        axes[3].set_title(f"Confidence-filtered union (conf >= {BEST_CONFIDENCE:.2f})")
        axes[4].imshow(final_mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[4].set_title(f"After: final mask (component area >= {MIN_COMPONENT_AREA})")
        show_component_overlay(axes[5], image, components)
        axes[5].set_title(f"After: {len(components)} labeled filament(s)")
        fig.suptitle(f"Validation postprocessing | {example['file_name']}", fontsize=13)
        finish_preview(fig, "postprocessing", example["file_name"])


def preview_test_inputs():
    paths = select_preview_items(get_test_image_paths())
    for path in paths:
        image = read_preview_image(path)
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        show_grayscale(axes[0], image)
        axes[0].set_title("Before: original test image")
        show_grayscale(axes[1], preprocess_grayscale(image))
        axes[1].set_title(f"After: {PREPROCESSING_METHOD} model input")
        fig.suptitle(f"Unseen test input | {path.name} | no ground truth", fontsize=13)
        finish_preview(fig, "test_inputs", path.name)
    return paths


def preview_submission_masks(output_path=SUBMISSION_PATH):
    # Read only the sampled sources' RLEs; show the masks actually written to CSV.
    paths = select_preview_items(get_test_image_paths())
    rows_by_source = {path.stem: [] for path in paths}
    with open(output_path, newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            source_stem = row["filament_id"].rsplit("_", 1)[0]
            if source_stem in rows_by_source:
                rows_by_source[source_stem].append(row)
    for path in paths:
        image = read_preview_image(path)
        rows = sorted(rows_by_source[path.stem], key=lambda row: int(row["filament_id"].rsplit("_", 1)[1]))
        components = [
            mask_utils.decode({"size": list(image.shape), "counts": row["segmentation_rle"].encode("ascii")}).astype(bool)
            for row in rows
        ]
        final_mask = np.zeros(image.shape, dtype=bool)
        for component in components:
            final_mask |= component
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        axes = axes.ravel()
        show_grayscale(axes[0], image)
        axes[0].set_title("Original test image")
        show_grayscale(axes[1], preprocess_grayscale(image))
        axes[1].set_title(f"Model input: {PREPROCESSING_METHOD}")
        axes[2].imshow(final_mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        axes[2].set_title("Final submitted mask (decoded from CSV RLE)")
        show_component_overlay(axes[3], image, components)
        axes[3].set_title(f"Submitted overlay: {len(rows)} filament row(s)")
        fig.suptitle(f"Submission verification | {path.name} | no test ground truth", fontsize=13)
        finish_preview(fig, "submission", path.name)


### Five-image preview: annotators before merge, merged masks and training-target overlays

Different colors identify separate annotator records. The union and the final YOLO polygon target are labeled separately so the existing component filtering is visible.


In [ ]:
preview_annotator_merges()


### Five-image preview: preprocessing before and after

Each source is shown with all four intensity methods at a fixed grayscale range. The bottom row overlays the same merged mask on the original and the selected processed image.


In [ ]:
preview_preprocessing()


## 5. Train YOLO26s-seg at full resolution

Training uses both available GPUs with a global batch of 2 (one 2048 x 2048 image per T4). Data loading uses two workers per GPU, four total. Disk caching is enabled only when the actual working volume can fit the estimated cache and reserve. If it cannot, caching is disabled (never switched to RAM).


In [ ]:
print("Building pretrained YOLO26s segmentation model...")
model = YOLO(YOLO_MODEL_NAME)

require_ram_headroom("training", minimum_gib=RAM_RESERVE_GIB + 6.0)
training_start = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=TRAIN_DEVICE,
    workers=YOLO_WORKERS_PER_GPU,
    cache=YOLO_CACHE_MODE,
    max_det=MAX_DETECTIONS,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",  # CHANGED FROM "auto" (Mandatory to unlock custom LR)
    lr0=0.001,  # ADDED: Initial learning rate (1e-3 is ideal for Adam/AdamW fine-tuning)
    lrf=0.01,  # ADDED: Final learning rate drops linearly to 1% of lr0 (0.00001)
    cos_lr=True,
    amp=(DEVICE.type == "cuda"),
    seed=SEED,
    deterministic=True,
    fliplr=0.5,
    flipud=0.5,
    degrees=180.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    plots=True,
    verbose=True,
)

training_time = time.perf_counter() - training_start

try:
    run_dir = Path(model.trainer.save_dir)
except AttributeError:
    run_dir = RUNS_DIR / RUN_NAME

best_checkpoint = run_dir / "weights" / "best.pt"
if not best_checkpoint.is_file():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_checkpoint}")

del train_results
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(str(best_checkpoint))

print(
    f"\n[SUCCESS] Training complete!\nRun directory: {run_dir}\nBest checkpoint: {best_checkpoint}\nTraining time: {training_time / 60:.1f} min",
    flush=True,
)

## Show Training History


In [ ]:
yolo_visual_groups = {
    "Overall Performance & Loss Tracking": ["results.png"],
    "Bounding Box Localization Metrics": [
        "BoxF1_curve.png",
        "BoxPR_curve.png",
        "BoxP_curve.png",
        "BoxR_curve.png",
    ],
    "Instance Segmentation Mask Metrics": [
        "MaskF1_curve.png",
        "MaskPR_curve.png",
        "MaskP_curve.png",
        "MaskR_curve.png",
    ],
    "Confusion Matrices": ["confusion_matrix.png", "confusion_matrix_normalized.png"],
    "Labels": ["labels.jpg"],
    "Training Dataset Batches & Anchors": [
        "train_batch0.jpg",
        "train_batch1.jpg",
        "train_batch2.jpg",
    ],
    "Validation Predictions vs Ground Truth": [
        "val_batch0_labels.jpg",
        "val_batch0_pred.jpg",
        "val_batch1_labels.jpg",
        "val_batch1_pred.jpg",
        "val_batch2_labels.jpg",
        "val_batch2_pred.jpg",
    ],
}


display_settings = {
    "Overall Performance & Loss Tracking": {
        "cols": 1,
        "col_width": 16,
        "row_height": 9,
    },
    "Bounding Box Localization Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Instance Segmentation Mask Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Confusion Matrices": {
        "cols": 1,
        "col_width": 10,
        "row_height": 8,
    },
    "Labels": {
        "cols": 1,
        "col_width": 12,
        "row_height": 8,
    },
    "Training Dataset Batches & Anchors": {
        "cols": 3,
        "col_width": 6,
        "row_height": 5,
    },
    "Validation Predictions vs Ground Truth": {
        "cols": 2,
        "col_width": 6,
        "row_height": 5,
    },
}


print(f"Loading generated artifact visuals from: {run_dir}\n" + "=" * 70)


for group_name, plot_files in yolo_visual_groups.items():
    valid_files = [f for f in plot_files if (run_dir / f).is_file()]

    if not valid_files:
        continue

    print(f"\n▶ Displaying Group: {group_name}")
    print("-" * 70)

    settings = display_settings[group_name]

    cols = settings["cols"]
    col_width = settings["col_width"]
    row_height = settings["row_height"]

    num_plots = len(valid_files)
    num_rows = (num_plots + cols - 1) // cols

    fig, axes = plt.subplots(
        num_rows,
        cols,
        figsize=(
            col_width * cols,
            row_height * num_rows,
        ),
    )

    axes = np.atleast_1d(axes).flatten()

    for idx, plot_name in enumerate(valid_files):
        img = cv2.imread(str(run_dir / plot_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[idx].imshow(img)
        axes[idx].set_title(
            plot_name,
            fontsize=11,
            fontweight="bold",
            pad=8,
        )
        axes[idx].axis("off")

    for idx in range(num_plots, len(axes)):
        axes[idx].axis("off")

    fig.tight_layout()
    plt.show()

In [ ]:
history_path = run_dir / "results.csv"

if not history_path.is_file():
    raise FileNotFoundError(f"YOLO training history not found: " f"{history_path}")

history = pd.read_csv(history_path)

history.columns = [column.strip() for column in history.columns]

print("Available history columns:")

print(", ".join(history.columns))

epochs = np.arange(
    1,
    len(history) + 1,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

for column, label in [
    (
        "train/seg_loss",
        "Train segmentation loss",
    ),
    (
        "val/seg_loss",
        "Validation segmentation loss",
    ),
]:
    if column in history.columns:
        axes[0].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[0].set(
    title="YOLO segmentation loss",
    xlabel="Epoch",
    ylabel="Loss",
)

axes[0].legend()
axes[0].grid(alpha=0.25)

for column, label in [
    (
        "metrics/mAP50(M)",
        "Mask mAP50",
    ),
    (
        "metrics/mAP50-95(M)",
        "Mask mAP50-95",
    ),
]:
    if column in history.columns:
        axes[1].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[1].set(
    title="YOLO validation mask metrics",
    xlabel="Epoch",
    ylabel="Metric",
    ylim=(0, 1),
)

axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

print(f"Best checkpoint: " f"{best_checkpoint}")

## 7. Prediction and metric helpers

The prediction path is shared by validation and submission:

1. selected grayscale intensity preprocessing -> 3 identical pretrained-model channels
2. YOLO instance segmentation
3. confidence filtering
4. union of selected masks
5. connected-component splitting
6. evaluation or RLE encoding


In [ ]:
def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")
    return cv2.cvtColor(preprocess_grayscale(image), cv2.COLOR_GRAY2BGR)


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"}
    ]
    return sorted(paths, key=lambda path: path.name)


def predict_instances_batch(images, min_confidence=0.15):
    sources = [to_yolo_image(image) for image in images]
    results = model.predict(
        source=sources,
        imgsz=IMAGE_SIZE,
        conf=min_confidence,
        iou=0.7,
        retina_masks=True,
        device=INFERENCE_DEVICE,
        batch=len(sources),
        max_det=MAX_DETECTIONS,
        verbose=False,
    )
    outputs = []

    for image, result in zip(images, results):
        if result.masks is None or result.boxes is None:
            outputs.append([])
            continue

        masks = result.masks.data.detach().float().cpu().numpy()
        confidences = result.boxes.conf.detach().float().cpu().numpy()
        native_height, native_width = image.shape
        instances = []

        for mask, confidence in zip(masks, confidences):
            if mask.shape != image.shape:
                mask = cv2.resize(
                    mask, (native_width, native_height), interpolation=cv2.INTER_NEAREST
                )
            instances.append({"mask": mask >= 0.5, "confidence": float(confidence)})
        outputs.append(instances)

    return outputs


def predict_instances(image, min_confidence=0.15):
    return predict_instances_batch([image], min_confidence=min_confidence)[0]


def union_instances(instances, confidence_threshold):
    selected = [
        instance
        for instance in instances
        if instance["confidence"] >= confidence_threshold
    ]
    if not selected:
        return None

    foreground = np.zeros_like(selected[0]["mask"], dtype=bool)
    for instance in selected:
        foreground |= instance["mask"]
    return foreground


def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8
    )
    components = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        components.append(labels == label)

    def component_key(component):
        ys, xs = np.where(component)
        return (
            -int(component.sum()),
            int(ys.min()) if len(ys) else 0,
            int(xs.min()) if len(xs) else 0,
        )

    return sorted(components, key=component_key)


def build_final_prediction_instances(instances, confidence_threshold):
    combined = union_instances(instances, confidence_threshold)
    if combined is None:
        return []
    return split_into_filaments(combined, MIN_COMPONENT_AREA)


def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")

## 8. Full-resolution validation

Validation inference processes one 2048 x 2048 image at a time. Its asynchronous CPU queue is capped at eight tasks, and each metric worker uses one thread. This limits simultaneous full-resolution arrays while still overlapping GPU inference with CPU metrics.


In [ ]:
dice_metric = DiceScore(
    num_classes=2, include_background=False, average="micro", input_format="index"
)


def calculate_pixel_metrics(prediction, target):
    prediction_tensor = torch.from_numpy(prediction.astype(np.int64)).unsqueeze(0)
    target_tensor = torch.from_numpy(target.astype(np.int64)).unsqueeze(0)

    prediction_has_foreground = bool(prediction.any())
    target_has_foreground = bool(target.any())

    if not prediction_has_foreground and not target_has_foreground:
        dice = 1.0
    else:
        dice = float(dice_metric(prediction_tensor, target_tensor).item())
        if np.isnan(dice):
            dice = 0.0

    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    true_positive = intersection

    false_positive = int(np.logical_and(prediction, ~target).sum())
    false_negative = int(np.logical_and(~prediction, target).sum())

    iou = intersection / union if union > 0 else 1.0
    precision_denominator = true_positive + false_positive
    recall_denominator = true_positive + false_negative

    precision = (
        true_positive / precision_denominator if precision_denominator > 0 else 1.0
    )
    recall = true_positive / recall_denominator if recall_denominator > 0 else 1.0

    return {
        "pixel_dice": float(dice),
        "pixel_iou": float(iou),
        "pixel_precision": float(precision),
        "pixel_recall": float(recall),
    }


def calculate_instance_metrics(
    prediction_instances, target_instances, iou_threshold=PQ_MATCH_IOU
):
    num_predictions = len(prediction_instances)
    num_targets = len(target_instances)

    if num_predictions == 0 and num_targets == 0:
        return {
            "pq": 1.0,
            "sq": 1.0,
            "rq": 1.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }
    if num_predictions == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": 0,
            "fn_instances": num_targets,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }
    if num_targets == 0:
        return {
            "pq": 0.0,
            "sq": 0.0,
            "rq": 0.0,
            "tp_instances": 0,
            "fp_instances": num_predictions,
            "fn_instances": 0,
            "one_to_many": 0,
            "many_to_one": 0,
            "matched_iou_sum": 0.0,
        }

    iou_matrix = np.zeros((num_targets, num_predictions), dtype=np.float32)

    for gt_index, gt_mask in enumerate(target_instances):
        gt_area = int(gt_mask.sum())
        if gt_area == 0:
            continue
        for pred_index, pred_mask in enumerate(prediction_instances):
            pred_area = int(pred_mask.sum())
            if pred_area == 0:
                continue
            intersection = int(np.logical_and(gt_mask, pred_mask).sum())
            if intersection == 0:
                continue
            union = gt_area + pred_area - intersection
            iou_matrix[gt_index, pred_index] = intersection / union

    gt_relation_counts = (iou_matrix > iou_threshold).sum(axis=1)
    pred_relation_counts = (iou_matrix > iou_threshold).sum(axis=0)
    one_to_many = int((gt_relation_counts > 1).sum())
    many_to_one = int((pred_relation_counts > 1).sum())

    candidate_pairs = []
    for gt_index in range(num_targets):
        for pred_index in range(num_predictions):
            iou = float(iou_matrix[gt_index, pred_index])
            if iou > iou_threshold:
                candidate_pairs.append((iou, gt_index, pred_index))

    candidate_pairs.sort(key=lambda item: item[0], reverse=True)
    matched_gt, matched_predictions, matched_ious = set(), set(), []

    for iou, gt_index, pred_index in candidate_pairs:
        if gt_index in matched_gt or pred_index in matched_predictions:
            continue
        matched_gt.add(gt_index)
        matched_predictions.add(pred_index)
        matched_ious.append(iou)

    tp = len(matched_ious)
    fp = num_predictions - tp
    fn = num_targets - tp
    matched_iou_sum = float(np.sum(matched_ious))

    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = matched_iou_sum / denominator if denominator > 0 else 0.0
    sq = matched_iou_sum / tp if tp > 0 else 0.0
    rq = tp / denominator if denominator > 0 else 0.0

    return {
        "pq": float(pq),
        "sq": float(sq),
        "rq": float(rq),
        "tp_instances": tp,
        "fp_instances": fp,
        "fn_instances": fn,
        "one_to_many": one_to_many,
        "many_to_one": many_to_one,
        "matched_iou_sum": matched_iou_sum,
    }

In [ ]:
def load_validation_images(rows):
    images, valid_rows = [], []
    for row in rows:
        image_path = TRAIN_IMAGE_DIR / row["file_name"]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"[WARNING] Could not read: {image_path}", flush=True)
            continue
        images.append(image)
        valid_rows.append(row)
    return valid_rows, images


def _worker_evaluate_validation_sample(task):
    configure_worker_process()
    global _local_val_coco

    if "_local_val_coco" not in globals():
        import sys

        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_val_coco = COCO(task["annotation_path"])
        sys.stdout = old_stdout

    image_id = task["image_id"]
    file_name = task["file_name"]
    image_shape = tuple(task["image_shape"])
    polygons = task["polygons"]
    confidences = task["confidences"]

    target_instances = []
    annotation_ids = _local_val_coco.getAnnIds(imgIds=[image_id])
    annotations = _local_val_coco.loadAnns(annotation_ids)

    for annotation in annotations:
        mask = _local_val_coco.annToMask(annotation).astype(bool)
        if mask.any():
            target_instances.append(mask)

    target_union = np.zeros(image_shape, dtype=bool)
    for target_instance in target_instances:
        target_union |= target_instance

    outputs = {}
    for threshold in CONFIDENCE_CANDIDATES:
        prediction_union = np.zeros(image_shape, dtype=np.uint8)
        selected_predictions = 0

        for polygon, confidence in zip(polygons, confidences):
            if confidence < threshold:
                continue
            polygon = np.asarray(polygon, dtype=np.float32)
            if len(polygon) < 3:
                continue
            cv2.fillPoly(prediction_union, [np.round(polygon).astype(np.int32)], 1)
            selected_predictions += 1

        prediction_union = prediction_union.astype(bool)
        prediction_instances = split_into_filaments(
            prediction_union, MIN_COMPONENT_AREA
        )
        pixel_metrics = calculate_pixel_metrics(prediction_union, target_union)
        instance_metrics = calculate_instance_metrics(
            prediction_instances, target_instances
        )

        outputs[threshold] = {
            "file_name": file_name,
            "pixel_dice": pixel_metrics["pixel_dice"],
            "pixel_iou": pixel_metrics["pixel_iou"],
            "pixel_precision": pixel_metrics["pixel_precision"],
            "pixel_recall": pixel_metrics["pixel_recall"],
            "pq": instance_metrics["pq"],
            "sq": instance_metrics["sq"],
            "rq": instance_metrics["rq"],
            "tp_instances": instance_metrics["tp_instances"],
            "fp_instances": instance_metrics["fp_instances"],
            "fn_instances": instance_metrics["fn_instances"],
            "predicted_instances": len(prediction_instances),
            "ground_truth_instances": len(target_instances),
            "one_to_many": instance_metrics["one_to_many"],
            "many_to_one": instance_metrics["many_to_one"],
            "selected_predictions": selected_predictions,
        }

    return outputs

In [ ]:
# Force garbage collection before allocating validation arrays
gc.collect()

require_ram_headroom("validation", minimum_gib=RAM_RESERVE_GIB + 2.0)
validation_start = time.perf_counter()

minimum_confidence = min(CONFIDENCE_CANDIDATES)

validation_examples = []
validation_preview_files = set()

threshold_results = {
    threshold: {"per_image": []} for threshold in CONFIDENCE_CANDIDATES
}

val_rows = val_records[["image_id", "file_name"]].to_dict(orient="records")

total_validation_records = len(val_rows)

print("\n" + "=" * 90)
print("                 PARALLEL VALIDATION")
print("=" * 90)
print(f"Validation records : {total_validation_records:,}")
print(f"CPU workers        : {NUM_WORKERS}")
print(f"Target aggregate CPU usage: ~{NUM_WORKERS * 100}%")
print(f"GPU batch size     : {VAL_BATCH_SIZE}")
print("CPU workload       : polygon rasterization + connected components + metrics")
print("=" * 90, flush=True)

# Persistent CPU pool.
pool = multiprocessing.Pool(
    processes=NUM_WORKERS,
    initializer=configure_worker_process,
)

pending = []

try:
    for batch_start in range(0, total_validation_records, VAL_BATCH_SIZE):
        batch_rows = val_rows[batch_start : batch_start + VAL_BATCH_SIZE]

        loaded_rows, images = load_validation_images(batch_rows)

        if not images:
            continue

        # ------------------------------------------------------
        # GPU inference in batches.
        # ------------------------------------------------------

        batch_instances = predict_instances_batch(
            images,
            min_confidence=minimum_confidence,
        )

        # ------------------------------------------------------
        # Submit CPU evaluation asynchronously.
        # ------------------------------------------------------

        for row, image, instances in zip(loaded_rows, images, batch_instances):
            polygons = []
            confidences = []

            for instance in instances:
                mask = instance["mask"]

                contours, _ = cv2.findContours(
                    mask.astype(np.uint8),
                    cv2.RETR_EXTERNAL,
                    cv2.CHAIN_APPROX_SIMPLE,
                )

                if not contours:
                    continue

                contour = max(contours, key=cv2.contourArea)

                if len(contour) < 3:
                    continue

                polygon = contour.reshape(-1, 2).astype(np.float32)

                polygons.append(polygon)
                confidences.append(float(instance["confidence"]))

            task = {
                "annotation_path": str(ANNOTATION_PATH),
                "image_id": row["image_id"],
                "file_name": row["file_name"],
                "image_shape": image.shape,
                "polygons": polygons,
                "confidences": confidences,
            }

            pending.append(
                pool.apply_async(
                    _worker_evaluate_validation_sample,
                    (task,),
                )
            )

            if len(pending) >= MAX_PENDING_VALIDATION:
                result = pending.pop(0).get()

                for threshold in CONFIDENCE_CANDIDATES:
                    threshold_results[threshold]["per_image"].append(result[threshold])

        # ------------------------------------------------------
        # Save a few examples for visualization.
        # ------------------------------------------------------

        # Cache up to PREVIEW_COUNT distinct physical images for before/after plots.
        if len(validation_examples) < PREVIEW_COUNT:
            for row, image, instances in zip(loaded_rows, images, batch_instances):
                if len(validation_examples) >= PREVIEW_COUNT:
                    break
                if row["file_name"] in validation_preview_files:
                    continue
                validation_preview_files.add(row["file_name"])

                annotations = coco.loadAnns(coco.getAnnIds(imgIds=[row["image_id"]]))
                target = np.zeros(image.shape, dtype=bool)

                for annotation in annotations:
                    target |= coco.annToMask(annotation).astype(bool)

                # Store detached copies to break cross-reference locks
                validation_examples.append(
                    {
                        "image": image.copy(),
                        "target": target.copy(),
                        "instances": [
                            {
                                k: (v.copy() if isinstance(v, np.ndarray) else v)
                                for k, v in inst.items()
                            }
                            for inst in instances
                        ],
                        "file_name": row["file_name"],
                    }
                )

        processed = min(
            batch_start + len(loaded_rows),
            total_validation_records,
        )

        print(
            f"{processed:,}/{total_validation_records:,} records | "
            f"pending CPU jobs: {len(pending)}",
            flush=True,
        )

        try:
            del loaded_rows, images, batch_instances, polygons, confidences, task
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(
        "\n[CPU] Draining remaining validation workers...",
        flush=True,
    )

    while pending:
        result = pending.pop(0).get()

        for threshold in CONFIDENCE_CANDIDATES:
            threshold_results[threshold]["per_image"].append(result[threshold])

finally:
    pool.close()
    pool.join()

validation_time = time.perf_counter() - validation_start

threshold_rows = []

for threshold in CONFIDENCE_CANDIDATES:
    per_image_df = pd.DataFrame(threshold_results[threshold]["per_image"])

    if per_image_df.empty:
        continue

    threshold_rows.append(
        {
            "confidence": threshold,
            "PQ": per_image_df["pq"].mean(),
            "SQ": per_image_df["sq"].mean(),
            "RQ": per_image_df["rq"].mean(),
            "Mean Dice": per_image_df["pixel_dice"].mean(),
            "Mean IoU": per_image_df["pixel_iou"].mean(),
            "Mean Precision": per_image_df["pixel_precision"].mean(),
            "Mean Recall": per_image_df["pixel_recall"].mean(),
            "TP": int(per_image_df["tp_instances"].sum()),
            "FP": int(per_image_df["fp_instances"].sum()),
            "FN": int(per_image_df["fn_instances"].sum()),
            "Predicted Instances": int(per_image_df["predicted_instances"].sum()),
            "GT Instances": int(per_image_df["ground_truth_instances"].sum()),
            "One-to-Many": int(per_image_df["one_to_many"].sum()),
            "Many-to-One": int(per_image_df["many_to_one"].sum()),
        }
    )

threshold_summary = (
    pd.DataFrame(threshold_rows)
    .sort_values("PQ", ascending=False)
    .reset_index(drop=True)
)

BEST_CONFIDENCE = float(threshold_summary.iloc[0]["confidence"])

print("\n" + "=" * 100)
print("                    VALIDATION RESULTS")
print("=" * 100)

display(threshold_summary.round(4))

print(f"\nSelected confidence threshold: {BEST_CONFIDENCE:.2f}")
print(f"Best validation PQ: {threshold_summary.iloc[0]['PQ']:.4f}")
print(f"Validation time: {validation_time:.1f}s")

print("=" * 100, flush=True)

### Five-image preview: validation postprocessing before and after

These are cached model predictions on five distinct validation images. Panels show the returned masks, confidence-filtered union, final area-filtered mask and labeled overlay. Ground truth is the original evaluation annotator, matching the unchanged metrics.


In [ ]:
preview_validation_postprocessing(validation_examples)


## 9. Generate the competition submission

Test inference runs one native-resolution image at a time. CPU connected-component extraction and RLE encoding use at most four single-threaded workers, and rows are written incrementally to the CSV.


### Five-image preview: test inputs before submission inference

Five evenly spaced test files show the original image and the selected preprocessing. Test ground truth is not used.


In [ ]:
test_preview_paths = preview_test_inputs()


In [ ]:
import queue


def _submission_worker_optimized(task, result_queue):
    """
    Worker process target. Processes the mask and pushes rows
    straight to a thread-safe queue to avoid main-thread stalling.
    """
    combined_mask = task["combined_mask"]
    path_stem = task["path_stem"]
    components = split_into_filaments(combined_mask, MIN_COMPONENT_AREA)

    rows = []
    for filament_counter, component in enumerate(components, start=1):
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))

    # Push the result back to the main thread
    result_queue.put(rows)


def create_submission_parallel(output_path=SUBMISSION_PATH):
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")

    require_ram_headroom("submission inference", minimum_gib=RAM_RESERVE_GIB + 2.0)
    total_images = len(paths)
    start_time = time.perf_counter()

    print(
        f"[START] Test images: {total_images:,}\n[CONFIG] GPU batch: {VAL_BATCH_SIZE}\n[CONFIG] CPU workers: {NUM_WORKERS}",
        flush=True,
    )

    # Use a Manager to create a thread/process-safe Queue
    manager = multiprocessing.Manager()
    result_queue = manager.Queue()

    cpu_pool = multiprocessing.Pool(
        processes=NUM_WORKERS, initializer=configure_worker_process
    )

    with open(output_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])

        total_rows_written = 0
        tasks_submitted = 0
        tasks_resolved = 0
        seen_filament_ids = set()

        try:
            for batch_start in range(0, total_images, VAL_BATCH_SIZE):
                batch_paths = paths[batch_start : batch_start + VAL_BATCH_SIZE]
                images = []

                for path in batch_paths:
                    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        raise OSError(f"Could not read test image: {path}")
                    images.append(image)

                # Heavy GPU inference step
                batch_instances = predict_instances_batch(
                    images, min_confidence=BEST_CONFIDENCE
                )

                for path, image, instances in zip(batch_paths, images, batch_instances):
                    combined = union_instances(instances, BEST_CONFIDENCE)
                    if combined is None:
                        combined = np.zeros(image.shape, dtype=bool)

                    # Dispatch to pool completely asynchronously
                    cpu_pool.apply_async(
                        _submission_worker_optimized,
                        args=(
                            {"combined_mask": combined, "path_stem": path.stem},
                            result_queue,
                        ),
                    )
                    tasks_submitted += 1

                # Flush out whatever CPU work has finished up to this point WITHOUT blocking
                while True:
                    try:
                        worker_rows = result_queue.get_nowait()
                        for fid, rle in worker_rows:
                            if fid in seen_filament_ids:
                                raise ValueError(
                                    f"Duplicate filament ID detected: {fid}"
                                )
                            seen_filament_ids.add(fid)
                            writer.writerow([fid, rle])
                            total_rows_written += 1
                        tasks_resolved += 1
                    except queue.Empty:
                        break  # No more completed tasks at this exact millisecond

                processed = min(batch_start + len(batch_paths), total_images)
                print(
                    f"Inference Progress: {processed:,}/{total_images:,} | Active CPU Tasks: {tasks_submitted - tasks_resolved}",
                    flush=True,
                )

                del images, batch_instances
                gc.collect()

            # Clean up the pool so workers don't accept new tasks
            cpu_pool.close()

            # Block and drain the remaining finished items out of the queue
            print("\nFinishing remaining CPU post-processing tasks...", flush=True)
            while tasks_resolved < tasks_submitted:
                worker_rows = (
                    result_queue.get()
                )  # Blocking get is safe here because pool is closing
                for fid, rle in worker_rows:
                    if fid in seen_filament_ids:
                        raise ValueError(f"Duplicate filament ID detected: {fid}")
                    seen_filament_ids.add(fid)
                    writer.writerow([fid, rle])
                    total_rows_written += 1
                tasks_resolved += 1

            cpu_pool.join()

        finally:
            # Safeguard to ensure pool terminates if an exception throws early
            cpu_pool.close()
            cpu_pool.terminate()
            cpu_pool.join()

    elapsed = time.perf_counter() - start_time
    print(
        f"\n[SUCCESS] Submission complete!\nProcessed: {total_images:,} images\nFilament rows written: {total_rows_written:,}\nRuntime: {elapsed / 60:.2f} min\nSaved: {output_path}",
        flush=True,
    )
    return pd.read_csv(output_path, nrows=5) if total_rows_written > 0 else None

## 10. Submission audit and final validation report


In [ ]:
if CREATE_SUBMISSION:
    create_submission_parallel()

    audit_columns = None
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    seen_ids = set()
    sample_rle_valid = False

    with open(SUBMISSION_PATH, newline="", encoding="utf-8") as audit_file:
        reader = csv.DictReader(audit_file)
        audit_columns = reader.fieldnames
        for row in reader:
            audit_row_count += 1
            if audit_row_count == 1:
                sample_rle_valid = bool(row.get("segmentation_rle"))

            if any(value is None for value in row.values()):
                audit_null_count += 1

            filament_id = row.get("filament_id")
            rle = row.get("segmentation_rle")
            if not filament_id or not rle:
                empty_rle_count += 1
            if filament_id in seen_ids:
                duplicate_ids += 1
            elif filament_id:
                seen_ids.add(filament_id)
else:
    audit_columns = ["filament_id", "segmentation_rle"]
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    sample_rle_valid = False

print("--- Submission File Integrity Audit (streamed) ---")
print(f"Total rows: {audit_row_count:,}")
print(f"Null values: {audit_null_count:,}")
print(f"Duplicate filament IDs: {duplicate_ids:,}")
print(f"Empty IDs/RLE strings: {empty_rle_count:,}")
print(f"Sample RLE check: {sample_rle_valid}")

submission_valid = (
    set(audit_columns or []) == {"filament_id", "segmentation_rle"}
    and audit_null_count == 0
    and duplicate_ids == 0
    and empty_rle_count == 0
    and (sample_rle_valid or audit_row_count == 0)
)

print(
    "\n"
    + (
        "[VERIFIED] Submission structure passed the local audit."
        if submission_valid
        else "[WARNING] Submission structure needs inspection."
    )
)

best_row = threshold_summary[threshold_summary["confidence"] == BEST_CONFIDENCE].iloc[0]

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "Panoptic Quality (PQ)",
            "Segmentation Quality (SQ)",
            "Recognition Quality (RQ)",
            "Mean Dice",
            "Mean IoU",
            "Mean Precision",
            "Mean Recall",
            "True Positives",
            "False Positives",
            "False Negatives",
            "Predicted Instances",
            "Ground-Truth Instances",
            "One-to-Many (Fragmentation)",
            "Many-to-One (Over-Merging)",
            "Selected Confidence",
        ],
        "Value": [
            best_row["PQ"],
            best_row["SQ"],
            best_row["RQ"],
            best_row["Mean Dice"],
            best_row["Mean IoU"],
            best_row["Mean Precision"],
            best_row["Mean Recall"],
            int(best_row["TP"]),
            int(best_row["FP"]),
            int(best_row["FN"]),
            int(best_row["Predicted Instances"]),
            int(best_row["GT Instances"]),
            int(best_row["One-to-Many"]),
            int(best_row["Many-to-One"]),
            BEST_CONFIDENCE,
        ],
    }
)

print("\n" + "=" * 75)
print("                 FINAL VALIDATION REPORT")
print("=" * 75)
display(final_metrics.round(4))

selected_per_image = pd.DataFrame(threshold_results[BEST_CONFIDENCE]["per_image"])
distribution = (
    selected_per_image[["pixel_dice", "pixel_iou", "pq", "sq", "rq"]]
    .agg(["mean", "median", "std", "min", "max"])
    .T
)
print("\n--- Per-image metric distribution ---")
display(distribution.round(4))

best_mask_map50 = (
    float(history["metrics/mAP50(M)"].max())
    if "metrics/mAP50(M)" in history.columns
    else np.nan
)
best_mask_map50_95 = (
    float(history["metrics/mAP50-95(M)"].max())
    if "metrics/mAP50-95(M)" in history.columns
    else np.nan
)

diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "YOLO best mask mAP50",
            "YOLO best mask mAP50-95",
            "Validation records",
            "Submission rows",
            "Submission valid",
        ],
        "Value": [
            best_mask_map50,
            best_mask_map50_95,
            len(selected_per_image),
            audit_row_count,
            submission_valid,
        ],
    }
)
print("\n--- Model / submission diagnostics ---")
display(diagnostics)

final_metrics.to_csv(FINAL_METRICS_PATH, index=False)
selected_per_image.to_csv(PER_IMAGE_METRICS_PATH, index=False)
print(f"Saved validation summary: {FINAL_METRICS_PATH}")
print(f"Saved per-image metrics: {PER_IMAGE_METRICS_PATH}")

### Five-image preview: masks written to the submission CSV

The saved RLEs for the same sampled test files are decoded and overlaid on the source images. Sources with no submitted filament rows are displayed with empty masks. No additional model prediction is run.


In [ ]:
if CREATE_SUBMISSION:
    preview_submission_masks()
else:
    print("Submission previews skipped because CREATE_SUBMISSION=False.")


## 11. Confidence threshold visualization

Threshold selection is based on **PQ**. Five-image postprocessing previews are shown immediately after validation.


In [ ]:
fig, axis = plt.subplots(figsize=(7, 4.5))
axis.plot(
    threshold_summary["confidence"],
    threshold_summary["PQ"],
    marker="o",
    label="Validation PQ",
)
axis.axvline(BEST_CONFIDENCE, linestyle="--", label=f"Selected: {BEST_CONFIDENCE:.2f}")
axis.set(
    title="Validation PQ vs confidence threshold",
    xlabel="YOLO instance confidence",
    ylabel="Panoptic Quality (PQ)",
    ylim=(0, 1),
)
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
plt.show()


## 12. Experiment summary

### Main outputs

All outputs below are inside `preprocessing_<method>_union/`.

- `yolo_filament_dataset/` - processed grayscale PNGs/merged labels and adaptive disk cache
- `yolo_runs/yolo26s_filament_full_resolution/weights/best.pt` - best checkpoint
- `solar_filament_yolo26s_submission.csv` - streamed competition submission
- `solar_filament_yolo26s_validation_metrics.csv` - validation summary
- `solar_filament_yolo26s_per_image_metrics.csv` - per-image metrics

### Preview galleries

`PREVIEW_COUNT = 5` controls the number of distinct images at each stage. If fewer
images are available, all are shown. Images are displayed inline and saved as PNGs:

- `previews/merged_masks/`: annotator outlines, union masks, overlays and saved YOLO training targets.
- `previews/preprocessing/`: raw/CLAHE/percentile/gamma, plus before/after merged-mask overlays.
- `previews/postprocessing/`: validation predictions before confidence filtering and after component filtering.
- `previews/test_inputs/`: five raw/processed test images without using test ground truth.
- `previews/submission/`: masks decoded from the actual CSV, with labeled overlays.

Plotting uses cached validation predictions and the existing submission CSV; no
extra GPU inference is required. Each figure is closed after display and saving.

### Resource plan

| Stage         | GPU                                         | CPU / memory control                                         |
| ------------- | ------------------------------------------- | ------------------------------------------------------------ |
| Training      | Both T4 GPUs when available; global batch 2 | 2 loader workers per GPU (4 total), AMP, adaptive disk cache |
| Dataset build | None                                        | 4 single-threaded workers; disk and RAM preflight            |
| Validation    | GPU 0                                       | Batch 1; at most 4 metric workers and 8 queued jobs          |
| Submission    | GPU 0                                       | Batch 1; 4 RLE workers; CSV written row by row               |

The disk preflight estimates a full RGB cache at 2048 x 2048, lossless grayscale PNGs for unique filenames, and training outputs. It enables disk caching only when that plan fits the actual working volume; otherwise it disables caching and preserves at least 4 GiB free. RAM checks stop expensive stages when available headroom is too low.
